# Module 2 Lab — Agent Risk Modeling & Autonomy Classification

**Scenario:** Enterprise Procurement Agent

This notebook builds a practical risk engine that converts an agent's capabilities into:

```text
Capability inventory
   ↓
Autonomy classification
   ↓
Risk dimensions
   ↓
Failure / threat scenarios
   ↓
Inherent risk
   ↓
Controls
   ↓
Residual risk
   ↓
Required control profile
```

The goal is **not** to invent a magical universal score. The goal is to make risk assumptions explicit, testable, and actionable.

## Tooling used

- **Pydantic** — typed risk contracts
- **pandas / NumPy** — analysis and risk matrices
- **NetworkX** — blast-radius / dependency graph
- **matplotlib** — sensitivity visualization
- **OpenAI SDK** — optional structured extraction of risk scenarios
- **PyRIT** — preview for adversarial validation in later modules

Primary references: NIST AI RMF, ISO/IEC 23894, NIST AI 100-2e2025, OWASP Agentic Top 10 2026, MITRE ATLAS.

In [ ]:
%pip install -q "pydantic>=2" pandas numpy networkx matplotlib openai
print("Dependencies installed.")

In [ ]:
from __future__ import annotations
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os, json, math

pd.set_option("display.max_colwidth", 120)

## 1. Define autonomy and risk contracts

In [ ]:
class AutonomyLevel(str, Enum):
    INFORMATIONAL = "informational"
    ASSISTED = "assisted_execution"
    BOUNDED = "bounded_autonomy"
    HIGH = "high_autonomy"

class RiskTier(str, Enum):
    LOW = "low"
    MODERATE = "moderate"
    HIGH = "high"
    CRITICAL = "critical"

class Capability(BaseModel):
    name: str
    description: str
    mutates_state: bool = False
    external_communication: bool = False
    privileged_access: bool = False
    financial_action: bool = False
    can_delegate: bool = False
    persistent_memory: bool = False
    human_approval_required: bool = False
    max_impact_usd: float = 0

class RiskDimensions(BaseModel):
    autonomy: float = Field(ge=0, le=1)
    impact: float = Field(ge=0, le=1)
    access: float = Field(ge=0, le=1)
    irreversibility: float = Field(ge=0, le=1)
    uncertainty: float = Field(ge=0, le=1)
    blast_radius: float = Field(ge=0, le=1)

class RiskAssessment(BaseModel):
    scenario: str
    dimensions: RiskDimensions
    inherent_score: float
    inherent_tier: RiskTier
    controls: list[str] = []
    control_effectiveness: float = Field(default=0, ge=0, le=1)
    residual_score: float = 0
    residual_tier: RiskTier = RiskTier.LOW

## 2. Build the procurement-agent capability inventory

In [ ]:
capabilities = [
    Capability(
        name="search_catalog",
        description="Search internal approved catalogue",
    ),
    Capability(
        name="read_vendor_records",
        description="Read confidential vendor data",
        privileged_access=True,
    ),
    Capability(
        name="email_supplier",
        description="Send supplier communication",
        mutates_state=True,
        external_communication=True,
    ),
    Capability(
        name="create_purchase_order",
        description="Create binding purchase order",
        mutates_state=True,
        financial_action=True,
        max_impact_usd=25_000,
        human_approval_required=True,
    ),
    Capability(
        name="issue_payment",
        description="Initiate vendor payment",
        mutates_state=True,
        privileged_access=True,
        financial_action=True,
        max_impact_usd=100_000,
        human_approval_required=True,
    ),
    Capability(
        name="delegate_research",
        description="Delegate research to a sub-agent",
        can_delegate=True,
    ),
    Capability(
        name="remember_vendor_preferences",
        description="Persist vendor preferences and task history",
        persistent_memory=True,
    ),
]
cap_df = pd.DataFrame([c.model_dump() for c in capabilities])
display(cap_df)

## 3. Autonomy classifier

A practical classifier should ask about **authority**, not whether the application happens to use an agent framework.

We use explainable rules here so learners can inspect the result.

In [ ]:
class AutonomySignals(BaseModel):
    recommends_only: bool = False
    prepares_actions: bool = False
    executes_without_approval: bool = False
    plans_multi_step: bool = False
    can_choose_tools: bool = False
    can_delegate: bool = False
    determines_completion: bool = False

def classify_autonomy(s: AutonomySignals) -> tuple[AutonomyLevel, list[str]]:
    reasons = []
    if s.executes_without_approval and (s.can_delegate or s.determines_completion):
        reasons.append("Can execute independently and control/delegate workflow.")
        return AutonomyLevel.HIGH, reasons
    if s.executes_without_approval:
        reasons.append("Can execute some actions without human approval.")
        return AutonomyLevel.BOUNDED, reasons
    if s.prepares_actions or s.plans_multi_step or s.can_choose_tools:
        reasons.append("Prepares/plans actions but execution remains human-gated.")
        return AutonomyLevel.ASSISTED, reasons
    reasons.append("Produces recommendations; a human performs the action.")
    return AutonomyLevel.INFORMATIONAL, reasons

In [ ]:
procurement_signals = AutonomySignals(
    prepares_actions=True,
    executes_without_approval=True,
    plans_multi_step=True,
    can_choose_tools=True,
    can_delegate=True,
    determines_completion=False,
)

autonomy_level, reasons = classify_autonomy(procurement_signals)
print("Autonomy level:", autonomy_level.value)
print("Reasons:", reasons)

## 4. Convert autonomy classification into a normalized risk dimension

In [ ]:
AUTONOMY_SCORE = {
    AutonomyLevel.INFORMATIONAL: 0.15,
    AutonomyLevel.ASSISTED: 0.35,
    AutonomyLevel.BOUNDED: 0.65,
    AutonomyLevel.HIGH: 0.90,
}
AUTONOMY_SCORE[autonomy_level]

## 5. Blast-radius graph with NetworkX

We model what the agent can reach.

This is deliberately simplified, but it reveals a critical point: **risk depends on downstream systems and trust zones**.

In [ ]:
G = nx.DiGraph()

nodes = {
    "Procurement Agent": {"criticality": 0.4},
    "Vendor DB": {"criticality": 0.5},
    "Email Gateway": {"criticality": 0.4},
    "Purchase API": {"criticality": 0.7},
    "ERP": {"criticality": 0.8},
    "Finance": {"criticality": 1.0},
    "Research Agent": {"criticality": 0.35},
    "Web": {"criticality": 0.25},
}

for node, attrs in nodes.items():
    G.add_node(node, **attrs)

edges = [
    ("Procurement Agent", "Vendor DB"),
    ("Procurement Agent", "Email Gateway"),
    ("Procurement Agent", "Purchase API"),
    ("Purchase API", "ERP"),
    ("ERP", "Finance"),
    ("Procurement Agent", "Research Agent"),
    ("Research Agent", "Web"),
]
G.add_edges_from(edges)

reachable = nx.descendants(G, "Procurement Agent")
blast = sum(G.nodes[n]["criticality"] for n in reachable) / sum(v["criticality"] for v in nodes.values())
blast = min(1.0, blast)

print("Reachable systems:", sorted(reachable))
print("Normalized blast-radius score:", round(blast, 3))

In [ ]:
plt.figure(figsize=(10,6))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(
    G, pos,
    with_labels=True,
    node_size=2600,
    arrows=True,
    font_size=9,
)
plt.title("Procurement Agent — Reachable Systems")
plt.axis("off")
plt.show()

## 6. Build a multidimensional risk profile

Keep the dimensions visible. Do not reduce the discussion to the aggregate score.

In [ ]:
def aggregate_risk(d: RiskDimensions) -> float:
    weights = {
        "autonomy": 0.18,
        "impact": 0.22,
        "access": 0.18,
        "irreversibility": 0.16,
        "uncertainty": 0.12,
        "blast_radius": 0.14,
    }
    values = d.model_dump()
    return round(sum(values[k] * w for k, w in weights.items()), 3)

def tier(score: float) -> RiskTier:
    if score < 0.30:
        return RiskTier.LOW
    if score < 0.50:
        return RiskTier.MODERATE
    if score < 0.72:
        return RiskTier.HIGH
    return RiskTier.CRITICAL

In [ ]:
po_risk = RiskDimensions(
    autonomy=AUTONOMY_SCORE[autonomy_level],
    impact=0.65,
    access=0.65,
    irreversibility=0.70,
    uncertainty=0.25,
    blast_radius=blast,
)
score = aggregate_risk(po_risk)
print(po_risk.model_dump())
print("Aggregate:", score, "Tier:", tier(score).value)

## 7. Capability-specific risk

Different tools owned by the same agent can have different risk.

That is why governance controls should often apply at the **capability/action level**, not just the agent level.

In [ ]:
def capability_risk(c: Capability, autonomy: AutonomyLevel, blast_radius: float) -> RiskDimensions:
    impact = min(1.0, c.max_impact_usd / 100_000)
    if c.mutates_state and impact < 0.3:
        impact = 0.3
    access = 0.8 if c.privileged_access else (0.5 if c.external_communication else 0.25)
    irreversibility = 0.8 if c.external_communication or c.financial_action else (0.45 if c.mutates_state else 0.1)
    uncertainty = 0.35 if c.can_delegate or c.persistent_memory else 0.2
    return RiskDimensions(
        autonomy=AUTONOMY_SCORE[autonomy],
        impact=impact,
        access=access,
        irreversibility=irreversibility,
        uncertainty=uncertainty,
        blast_radius=blast_radius if c.privileged_access or c.mutates_state else blast_radius * 0.35,
    )

rows=[]
for c in capabilities:
    d=capability_risk(c, autonomy_level, blast)
    s=aggregate_risk(d)
    rows.append({"capability":c.name, **d.model_dump(), "score":s, "tier":tier(s).value})
cap_risk_df=pd.DataFrame(rows).sort_values("score",ascending=False)
display(cap_risk_df)

## 8. FMEA-style failure analysis

We use three common ordinal dimensions:

- **Severity**
- **Occurrence**
- **Detectability**

Higher detectability score means **harder to detect**, matching common FMEA conventions.

The `RPN = S × O × D` is used only for prioritization.

In [ ]:
class FailureMode(BaseModel):
    id: str
    capability: str
    failure: str
    effect: str
    severity: int = Field(ge=1, le=10)
    occurrence: int = Field(ge=1, le=10)
    detectability: int = Field(ge=1, le=10)
    adversarial: bool = False
    controls: list[str] = []

    @property
    def rpn(self) -> int:
        return self.severity * self.occurrence * self.detectability

failure_modes = [
    FailureMode(
        id="F-01",
        capability="create_purchase_order",
        failure="Agent selects stale/unapproved vendor",
        effect="Policy violation or financial loss",
        severity=8, occurrence=4, detectability=5,
        controls=["approved vendor filter", "runtime authorization"],
    ),
    FailureMode(
        id="F-02",
        capability="create_purchase_order",
        failure="Duplicate purchase order during retry/loop",
        effect="Duplicate financial commitment",
        severity=8, occurrence=3, detectability=4,
        controls=["idempotency key", "tool-call budget"],
    ),
    FailureMode(
        id="F-03",
        capability="remember_vendor_preferences",
        failure="Obsolete exception persists in memory",
        effect="Future decisions violate current policy",
        severity=6, occurrence=5, detectability=7,
        controls=["memory TTL", "provenance", "validation"],
    ),
    FailureMode(
        id="A-01",
        capability="email_supplier",
        failure="Indirect prompt injection hijacks procurement goal",
        effect="Unauthorized data/action path",
        severity=9, occurrence=4, detectability=7,
        adversarial=True,
        controls=["untrusted-content boundary", "policy enforcement"],
    ),
]

fmea_df = pd.DataFrame([
    {**f.model_dump(), "rpn": f.rpn} for f in failure_modes
]).sort_values("rpn",ascending=False)
display(fmea_df[["id","capability","failure","severity","occurrence","detectability","adversarial","rpn"]])

### Important

FMEA and adversarial threat modeling should complement each other.

- FMEA helps with **failure**.
- OWASP / MITRE ATLAS help with **motivated adversaries**.

## 9. OWASP / MITRE threat mapping

In [ ]:
threat_map = pd.DataFrame([
    {
        "scenario": "Supplier document contains malicious instructions",
        "owasp_agentic": "Agent Goal Hijacking / Tool Misuse",
        "mitre_atlas_focus": "Prompt injection / agentic execution path",
        "control": "Treat retrieved content as untrusted + external policy"
    },
    {
        "scenario": "Agent inherits broad finance credentials",
        "owasp_agentic": "Identity & Privilege Abuse",
        "mitre_atlas_focus": "Credential access / privilege escalation",
        "control": "Task-scoped authority + least privilege"
    },
    {
        "scenario": "Compromised third-party tool server",
        "owasp_agentic": "Agentic Supply Chain Vulnerability",
        "mitre_atlas_focus": "Initial access / execution / persistence",
        "control": "Tool provenance + allowlist + isolation"
    },
    {
        "scenario": "Research agent causes repeated downstream actions",
        "owasp_agentic": "Cascading Failure / Tool Misuse",
        "mitre_atlas_focus": "Impact",
        "control": "Delegation depth + budgets + idempotency"
    },
])
display(threat_map)

## 10. Inherent vs. residual risk

In [ ]:
CONTROL_STRENGTH = {
    "output_evaluation": 0.08,
    "human_approval": 0.18,
    "least_privilege": 0.16,
    "runtime_policy": 0.18,
    "idempotency": 0.12,
    "continuous_evaluation": 0.10,
    "red_teaming": 0.08,
    "strong_identity": 0.12,
}

def residual_score(inherent: float, controls: list[str]) -> tuple[float,float]:
    # Simplified, capped additive effectiveness for teaching.
    effectiveness = min(0.75, sum(CONTROL_STRENGTH.get(c, 0) for c in controls))
    residual = round(inherent * (1 - effectiveness), 3)
    return residual, effectiveness

inherent = aggregate_risk(po_risk)
controls = ["human_approval","least_privilege","runtime_policy","continuous_evaluation","strong_identity"]
residual, effectiveness = residual_score(inherent, controls)

print("Inherent:", inherent, tier(inherent).value)
print("Control effectiveness:", round(effectiveness,3))
print("Residual:", residual, tier(residual).value)

## 11. Risk tier → minimum control profile

A risk process should end with **engineering requirements**, not only a label.

In [ ]:
CONTROL_PROFILES = {
    RiskTier.LOW: [
        "output evaluation",
        "data access controls",
        "basic telemetry",
    ],
    RiskTier.MODERATE: [
        "human approval for state changes",
        "tool allowlist",
        "traceability",
        "schema validation",
    ],
    RiskTier.HIGH: [
        "strong agent identity",
        "least privilege",
        "runtime authorization/policy",
        "risk-based escalation",
        "continuous evaluation",
        "incident containment",
    ],
    RiskTier.CRITICAL: [
        "reduce autonomy by default",
        "task-scoped credentials",
        "policy-as-code",
        "multi-party approval",
        "complete trajectory evidence",
        "continuous red teaming",
        "read-only/restrict/disable modes",
    ],
}

def recommended_controls(risk_tier: RiskTier):
    return CONTROL_PROFILES[risk_tier]

for t in RiskTier:
    print("\n", t.value.upper())
    for control in recommended_controls(t):
        print(" -", control)

## 12. Generate a risk assessment object

In [ ]:
assessment = RiskAssessment(
    scenario="Procurement agent creates a purchase order",
    dimensions=po_risk,
    inherent_score=inherent,
    inherent_tier=tier(inherent),
    controls=controls,
    control_effectiveness=effectiveness,
    residual_score=residual,
    residual_tier=tier(residual),
)
print(assessment.model_dump_json(indent=2))

## 13. Sensitivity analysis with NumPy

A useful risk model should reveal which assumptions drive the result.

We vary:

- impact
- uncertainty
- blast radius

rather than treating the initial numbers as facts.

In [ ]:
samples = []
rng = np.random.default_rng(42)

for _ in range(5000):
    d = RiskDimensions(
        autonomy=po_risk.autonomy,
        impact=float(np.clip(rng.normal(po_risk.impact, 0.12),0,1)),
        access=po_risk.access,
        irreversibility=po_risk.irreversibility,
        uncertainty=float(np.clip(rng.normal(po_risk.uncertainty, 0.10),0,1)),
        blast_radius=float(np.clip(rng.normal(po_risk.blast_radius, 0.10),0,1)),
    )
    samples.append(aggregate_risk(d))

samples=np.array(samples)
print("Mean:", samples.mean().round(3))
print("5th–95th percentile:", np.quantile(samples,[0.05,0.95]).round(3))

In [ ]:
plt.figure(figsize=(9,5))
plt.hist(samples, bins=35)
plt.axvline(inherent, linestyle="--", label="baseline")
plt.xlabel("Aggregate risk score")
plt.ylabel("Simulation count")
plt.title("Risk Sensitivity — Assumption Uncertainty")
plt.legend()
plt.show()

### Interpretation

If a small change in assumptions moves the system between risk tiers, the correct response is usually **more evidence**, not a debate over the second decimal place.

## 14. Control what-if analysis

In [ ]:
what_if = []
control_sets = [
    [],
    ["human_approval"],
    ["human_approval","runtime_policy"],
    ["human_approval","runtime_policy","least_privilege"],
    ["human_approval","runtime_policy","least_privilege","strong_identity","continuous_evaluation"],
]
for cs in control_sets:
    r,e = residual_score(inherent, cs)
    what_if.append({
        "controls": ", ".join(cs) or "(none)",
        "effectiveness":e,
        "residual_score":r,
        "residual_tier":tier(r).value,
    })
display(pd.DataFrame(what_if))

## 15. Risk acceptance decision

Risk classification should connect to governance decisions.

Example policy:

- **Low** — approve through standard process
- **Moderate** — approve with named controls
- **High** — architecture/security review required
- **Critical** — autonomy must be reduced unless exception is explicitly approved

In [ ]:
def governance_disposition(t: RiskTier) -> str:
    return {
        RiskTier.LOW: "STANDARD_APPROVAL",
        RiskTier.MODERATE: "APPROVE_WITH_CONTROLS",
        RiskTier.HIGH: "SPECIALIST_REVIEW_REQUIRED",
        RiskTier.CRITICAL: "REDUCE_AUTONOMY_OR_EXECUTIVE_EXCEPTION",
    }[t]

print("Residual disposition:", governance_disposition(assessment.residual_tier))

## 16. Export an enterprise risk register

In [ ]:
risk_register = pd.DataFrame([
    {
        "risk_id": f.id,
        "scenario": f.failure,
        "capability": f.capability,
        "effect": f.effect,
        "adversarial": f.adversarial,
        "severity": f.severity,
        "likelihood": f.occurrence,
        "detectability": f.detectability,
        "rpn": f.rpn,
        "controls": "; ".join(f.controls),
        "owner": "Procurement AI Platform",
        "status": "open",
    }
    for f in failure_modes
])

risk_register.to_csv("module02_agent_risk_register.csv", index=False)
display(risk_register)
print("Saved module02_agent_risk_register.csv")

# 17. Optional state-of-the-art extension — LLM-assisted risk scenario extraction

An LLM can help generate **candidate** failure scenarios from an architecture description.

It should **not** be the final risk decision-maker.

Recommended workflow:

```text
Architecture
  ↓
LLM generates candidate scenarios
  ↓
Pydantic structured output
  ↓
Human/security review
  ↓
NIST / OWASP / ATLAS mapping
  ↓
Approved risk register
```

Use the OpenAI SDK only if `OPENAI_API_KEY` is configured.

In [ ]:
from pydantic import BaseModel

class CandidateRisk(BaseModel):
    title: str
    trigger: str
    consequence: str
    category: str
    recommended_review: str

class CandidateRiskList(BaseModel):
    risks: list[CandidateRisk]

In [ ]:
if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()

    architecture = '''
    A procurement agent can retrieve vendor records, search catalogues,
    email external suppliers, create purchase orders, delegate research
    to a sub-agent, and store persistent vendor preferences.
    Purchase orders above $5,000 require human approval.
    '''

    response = client.responses.parse(
        model=os.getenv("OPENAI_MODEL", "gpt-5.6-sol"),
        input=[
            {
                "role":"system",
                "content":"Generate candidate enterprise AI-agent risk scenarios. Do not make final risk decisions."
            },
            {"role":"user","content":architecture},
        ],
        text_format=CandidateRiskList,
    )
    display(pd.DataFrame([r.model_dump() for r in response.output_parsed.risks]))
else:
    print("Set OPENAI_API_KEY to run the optional structured risk discovery section.")

## 18. PyRIT preview

Once a scenario is identified, red-team tools can test whether it is exploitable.

Microsoft PyRIT supports:

- scenarios,
- attacks,
- targets,
- scorers,
- batch scoring.

Its current scoring model supports boolean and normalized 0–1 score types.

In the later security/red-team module, convert risks such as:

> “Indirect prompt injection causes unauthorized vendor selection”

into executable PyRIT attack scenarios and measure actual attack success.

Documentation:
https://microsoft.github.io/PyRIT/latest/code/scoring/scoring/

## 19. Risk-regression tests

In [ ]:
def require_control(t: RiskTier, control: str):
    assert control in CONTROL_PROFILES[t], f"{control} is missing from {t.value} profile"

require_control(RiskTier.HIGH, "runtime authorization/policy")
require_control(RiskTier.CRITICAL, "multi-party approval")

assert classify_autonomy(AutonomySignals(recommends_only=True))[0] == AutonomyLevel.INFORMATIONAL
assert classify_autonomy(AutonomySignals(prepares_actions=True))[0] == AutonomyLevel.ASSISTED
assert classify_autonomy(AutonomySignals(executes_without_approval=True))[0] == AutonomyLevel.BOUNDED
assert classify_autonomy(
    AutonomySignals(executes_without_approval=True, can_delegate=True)
)[0] == AutonomyLevel.HIGH

print("Risk-model regression tests passed.")

# 20. Exercises

### Exercise A — New capability

Add:

`delete_vendor_record`

Classify its:

- autonomy relevance
- impact
- access
- irreversibility
- uncertainty
- blast radius

### Exercise B — Reduce blast radius

Modify the graph so the procurement agent cannot reach Finance.

Measure the change.

### Exercise C — Memory risk

Add a failure mode:

> “Temporary CFO exception persists indefinitely.”

Propose prevention, detection, and recovery controls.

### Exercise D — Delegation risk

Give the research sub-agent access to vendor records.

How does this change:

- access,
- blast radius,
- privilege risk?

### Exercise E — Residual risk evidence

Do not assume `runtime_policy = 18% effective`.

Define the evaluation evidence you would need to justify a control-effectiveness estimate.

### Exercise F — Critical risk

Create a configuration that reaches the critical tier.

What is the safest response:

- add controls,
- reduce autonomy,
- remove a tool,
- add approval,
- redesign the workflow?

# 21. Key takeaways

1. Agent risk is multidimensional.
2. Autonomy is important, but **autonomy ≠ total risk**.
3. Capability-level assessment is often more useful than one agent-wide label.
4. Keep inherent failure and adversarial misuse distinct.
5. Use FMEA for operational failure modes.
6. Use OWASP and MITRE ATLAS for adversarial scenarios.
7. Blast radius should include downstream systems and delegated agents.
8. A numeric score supports judgment; it does not replace judgment.
9. Compare inherent and residual risk.
10. Risk tier must produce an engineering control profile.
11. Risk assumptions should be tested with telemetry, evaluation, and red teaming.
12. Higher autonomy should be earned through evidence.